<a href="https://colab.research.google.com/github/rgdeekshith/FIFA-World-Cup-EDA-Analysis/blob/main/FIFA_World_Cup_1930_2022_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots show inside the notebook
%matplotlib inline

# Set a basic plot style
sns.set(style="whitegrid")

In [ ]:
# Replace this filename if your CSV has a different name
csv_path = "/content/fifa_worldcup_1930_2022.csv"

# Read the CSV into a pandas DataFrame
df = pd.read_csv(csv_path, encoding='latin1')

# Show the first 5 rows
df.head()

# Information of the CSV
df.info()

In [ ]:
# Show all column names
df.columns.tolist()

# Basic shape: (rows, columns)
df.shape

In [ ]:
# 1. Convert Match Date to datetime
df["Match Date"] = pd.to_datetime(df["Match Date"], errors = "coerce")

# 2. Extract Year from Match Date
df["Year"] = df["Match Date"].dt.year

# 3. Create TotalGoals as sum of home and away goals
df["TotalGoals"] = df["Home Team Score"] + df["Away Team Score"]

# Look at the first 5 rows of the new columns
df[["Match Date", "Year", "Home Team Score", "Away Team Score", "TotalGoals"]].head()

In [ ]:
# Overall descriptive stats for TotalGoals
df["TotalGoals"].describe()

# Average goals per match by Year
goals_by_year = df.groupby("Year")["TotalGoals"].mean()

goals_by_year.head()

# Find the year with the highest average goals
max_year = goals_by_year.idxmax()
max_avg_goals = goals_by_year.max()

# Find the year with the lowest average goals
min_year = goals_by_year.idxmin()
min_avg_goals = goals_by_year.min()

# Create a small DataFrame with just these two rows
extreme_years = pd.DataFrame({
    "Year": [min_year, max_year],
    "AverageGoals": [min_avg_goals, max_avg_goals]
})
print(extreme_years)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(goals_by_year.index, goals_by_year.values, marker="o")
plt.title("Average Goals per Match by World Cup Year")
plt.xlabel("Year")
plt.ylabel("Average Goals per Match")
plt.grid(True)
plt.show()

In [ ]:
# Goals scored when playing as the home team
home_goals_by_team = df.groupby("Home Team Name")["Home Team Score"].sum()

# Goals scored when playing as the away team
away_goals_by_team = df.groupby("Away Team Name")["Away Team Score"].sum()

# Align and add them to get total goals per team
total_goals_by_team = home_goals_by_team.add(away_goals_by_team, fill_value=0)

# Convert to a DataFrame and sort by total goals descending
team_goals_df = total_goals_by_team.sort_values(ascending=False).reset_index()
team_goals_df.columns = ["Team", "TotalGoals"]

# Show top 10 teams by total goals
team_goals_df.head(10)

In [ ]:
# Wins as home team
home_wins = df.groupby("Home Team Name")["Home Team Win"].sum()

# Wins as away team
away_wins = df.groupby("Away Team Name")["Away Team Win"].sum()

# Total wins per team
total_wins_by_team = home_wins.add(away_wins, fill_value=0)

# Create DataFrame and sort
team_wins_df = total_wins_by_team.sort_values(ascending=False).reset_index()
team_wins_df.columns = ["Team", "TotalWins"]

# Show top 10 teams by total wins
team_wins_df.head(10)

In [ ]:
# Combine goals and wins into a single DataFrame
team_stats_df = pd.merge(
    team_goals_df,
    team_wins_df,
    on="Team",
    how="inner"
)

# Compute goals per win as a simple derived metric
team_stats_df["GoalsPerWin"] = team_stats_df["TotalGoals"] / team_stats_df["TotalWins"]

# Sort by total wins
team_stats_df_sorted = team_stats_df.sort_values("TotalWins", ascending=False)

team_stats_df_sorted.head(10)

In [ ]:
top10_wins = team_stats_df_sorted.head(10)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=top10_wins,
    x="Team",
    y="TotalWins",
    palette="Blues_d"
)
plt.title("Top 10 Teams by World Cup Match Wins")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Total Wins")
plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Matches as home team
home_matches = df.groupby("Home Team Name")["Match Id"].count()

# Matches as away team
away_matches = df.groupby("Away Team Name")["Match Id"].count()

# Total matches played per team
matches_played_by_team = home_matches.add(away_matches, fill_value=0)

# Convert to DataFrame
team_matches_df = matches_played_by_team.reset_index()
team_matches_df.columns = ["Team", "MatchesPlayed"]

team_matches_df.head(10)

In [ ]:
# Merge matches played into the existing team stats
team_stats_df = pd.merge(
    team_stats_df,
    team_matches_df,
    on="Team",
    how="inner"
)

# Compute Win Rate = TotalWins / MatchesPlayed
team_stats_df["WinRate"] = team_stats_df["TotalWins"] / team_stats_df["MatchesPlayed"]

# Sort by WinRate, but keep only teams with a reasonable number of matches (e.g., >= 10)
team_stats_with_enough_matches = team_stats_df[team_stats_df["MatchesPlayed"] >= 10]

top10_winrate = team_stats_with_enough_matches.sort_values(
    "WinRate",
    ascending=False
).head(10)

top10_winrate

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=top10_winrate,
    x="Team",
    y="WinRate",
    palette="Greens_d"
)
plt.title("Top 10 Teams by Win Rate (min 10 matches)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Win Rate")
plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
df[["Stage Name", "Group Stage", "Knockout Stage"]].head(10)

In [ ]:
# Average goals in group stage matches
group_stage_goals_mean = df.loc[df["Group Stage"] == 1, "TotalGoals"].mean()

# Average goals in knockout stage matches
knockout_stage_goals_mean = df.loc[df["Knockout Stage"] == 1, "TotalGoals"].mean()

group_stage_goals_mean, knockout_stage_goals_mean

In [ ]:
stage_goals_summary = pd.DataFrame({
    "StageType": ["Group Stage", "Knockout Stage"],
    "AverageGoals": [group_stage_goals_mean, knockout_stage_goals_mean]
})

stage_goals_summary

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(
    data=stage_goals_summary,
    x="StageType",
    y="AverageGoals",
    palette="Oranges_d"
)
plt.title("Average Goals per Match: Group vs Knockout")
plt.ylabel("Average Goals per Match")
plt.xlabel("")
plt.ylim(0, stage_goals_summary["AverageGoals"].max() + 0.5)
plt.show()

In [ ]:
!pip install -U ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

In [ ]:
profile = ProfileReport(df, title="FIFA World Cup Data Profiling Report", explorative=True)

In [ ]:
profile.to_file("FIFA_Analysis.html")